Q-12 You are working on a dataset(disease_risk.csv) containing multiple input
features related to patient health parameters and a binary target variable
Disease_Risk. Implement a program to perform Logistic Regression for
binary classification and analyze the relationship between the input
features and the target variable. Perform the following tasks:

i. Read the given dataset into a dataframe.

ii. Split the dataset into training and testing sets in the ratio 70:30.

iii. Train a Logistic Regression model using the input features to predict
the binary target variable Disease_Risk.

iv. Determine the model coefficients and intercept.

v. Evaluate the performance of the model using accuracy, and show the
confusion matrix.

vi. Use the trained model to predict the class label for new data points

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.feature_selection import SequentialFeatureSelector

In [43]:
df=pd.DataFrame(pd.read_csv("disease.csv"))
df.head()

,Age,Gender,BMI,BloodPressure,Glucose,Cholesterol,SmokingStatus,PhysicalActivity,FamilyHistory,DietQuality,Disease_Risk
0,26,Male,35.4,155,72,217,Former,High,Yes,Average,High
1,45,Male,22.5,164,115,260,Never,Moderate,No,Poor,High
2,71,Male,28.2,117,94,247,Never,Moderate,No,Poor,High
3,42,Female,22.9,135,95,206,Never,Low,No,Poor,Low
4,61,Female,20.9,107,197,286,Never,High,No,Average,High


In [44]:
le=LabelEncoder()
cols=['Gender','SmokingStatus','PhysicalActivity','FamilyHistory','DietQuality','Disease_Risk']
for i in cols:
  df[i]=le.fit_transform(df[i])
df.head()

,Age,Gender,BMI,BloodPressure,Glucose,Cholesterol,SmokingStatus,PhysicalActivity,FamilyHistory,DietQuality,Disease_Risk
0,26,1,35.4,155,72,217,1,0,1,0,0
1,45,1,22.5,164,115,260,2,2,0,2,0
2,71,1,28.2,117,94,247,2,2,0,2,0
3,42,0,22.9,135,95,206,2,1,0,2,1
4,61,0,20.9,107,197,286,2,0,0,0,0


In [45]:
x=df.drop('Disease_Risk',axis=1)
y=df['Disease_Risk']
y.value_counts()

,count
Disease_Risk,
0,332
1,68


In [46]:
model=LogisticRegression()

In [47]:
fs=SequentialFeatureSelector(model,n_features_to_select=5)
fs.fit(x,y)

SequentialFeatureSelector(estimator=LogisticRegression(),
                          n_features_to_select=5)

In [48]:
selected_features=x.columns[fs.get_support()]
selected_features

Index(['Age', 'BloodPressure', 'Glucose', 'Cholesterol', 'SmokingStatus'], dtype='object')

In [49]:
ss=StandardScaler()
x=df[selected_features]
df_model=pd.DataFrame(ss.fit_transform(x.drop("SmokingStatus",axis=1)),columns=x.columns[:-1])
df_model["SmokingStatus"]=df['SmokingStatus']
df_model.head()

,Age,BloodPressure,Glucose,Cholesterol,SmokingStatus
0,-1.266136,0.770535,-1.753334,-0.142442,1
1,-0.212640,1.116023,-0.574760,0.876269,2
2,1.228987,-0.688193,-1.150343,0.568287,2
3,-0.378981,0.002783,-1.122934,-0.403042,2
4,0.674515,-1.072069,1.672752,1.492234,2


In [50]:
x_train,x_test,y_train,y_test=train_test_split(df_model,y,test_size=0.3,random_state=42)

In [52]:
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

In [65]:
slope = model.coef_[0]
intercept = model.intercept_
print(f"Slope : {slope}\n Intercept : {intercept}")

Slope : [-0.7004387  -0.66713641 -1.78618716 -0.69522543  1.14698024]
 Intercept : [-4.43274447]


In [56]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Confusion Matrix:\n{conf_matrix}")

Accuracy: 0.8417
Precision: 0.6667
Recall: 0.2727
F1 Score: 0.3871
Confusion Matrix:
[[95  3]
 [16  6]]


In [60]:
new_patient = {
    "Age": 52,
    "Gender": "Male",
    "BMI": 29.5,
    "BloodPressure": 145,
    "Glucose": 160,
    "Cholesterol": 220,
    "SmokingStatus": "Current",
    "PhysicalActivity": "Low",
    "FamilyHistory": "Yes",
    "DietQuality": "Average"
}
test=pd.DataFrame([new_patient])
cols=['Gender','SmokingStatus','PhysicalActivity','FamilyHistory','DietQuality']
for i in cols:
  test[i]=le.fit_transform(test[i])
test

,Age,Gender,BMI,BloodPressure,Glucose,Cholesterol,SmokingStatus,PhysicalActivity,FamilyHistory,DietQuality
0,52,0,29.5,145,160,220,0,0,0,0


In [63]:
prediction=model.predict(test[selected_features])[0]
print("Prediction : ",end='')
if prediction:
  print("High")
else:
  print("Low")

Prediction : Low
